# Notebook 01 — Define Allen Lab Context

This notebook validates the initial ChatGPT lab-context profile for `allen-lab-report-tool`.

The goal is not to generate a full report yet. The goal is to make the Allen Lab context inspectable, exportable, and review-ready before connecting it to papers, datasets, or source documents.

## Outputs

- `results/chatgpt/allen_lab_context.json`
- `reports/chatgpt/allen_lab_context.md`


## 1. Setup

This assumes the notebook is run from the repo root, or from Colab after cloning the repo.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()

# If running from notebooks/chatgpt, move two levels up.
if REPO_ROOT.name == "chatgpt" and REPO_ROOT.parent.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parents[1]

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

RESULTS_DIR = REPO_ROOT / "results" / "chatgpt"
REPORTS_DIR = REPO_ROOT / "reports" / "chatgpt"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Source path:", SRC_PATH)
print("Results dir:", RESULTS_DIR)
print("Reports dir:", REPORTS_DIR)


## 2. Import ChatGPT lab context

This imports your existing `src/chatgpt/lab_context.py` file.


In [ ]:
from chatgpt.lab_context import ALLEN_LAB_CONTEXT

ALLEN_LAB_CONTEXT


## 3. Validate required fields

This keeps Notebook 01 simple: it checks whether the context profile contains the fields later notebooks will expect.


In [ ]:
required_keys = [
    "institution",
    "likely_focus_areas",
    "likely_equipment_or_platforms",
    "report_priorities",
]

missing = [key for key in required_keys if key not in ALLEN_LAB_CONTEXT]

if missing:
    raise ValueError(f"Missing required context keys: {missing}")

for key in required_keys:
    print(f"✓ {key}: {type(ALLEN_LAB_CONTEXT[key]).__name__}")


## 4. Display context tables

The tables make the assumptions readable before exporting them.


In [ ]:
import pandas as pd

focus_df = pd.DataFrame({"likely_focus_area": ALLEN_LAB_CONTEXT["likely_focus_areas"]})
equipment_df = pd.DataFrame({"likely_equipment_or_platform": ALLEN_LAB_CONTEXT["likely_equipment_or_platforms"]})
priorities_df = pd.DataFrame({"report_priority": ALLEN_LAB_CONTEXT["report_priorities"]})

focus_df, equipment_df, priorities_df


In [ ]:
display(focus_df)
display(equipment_df)
display(priorities_df)


## 5. Add provenance-aware review notes

The equipment/platform list is useful, but should be clearly marked as inferred until confirmed from a specific Allen Lab page, paper, methods section, or dataset description.


In [ ]:
PROVENANCE_NOTES = [
    "Institution and broad focus areas are used as a context profile for report generation.",
    "Equipment/platform entries are treated as likely or plausible until confirmed against specific Allen Institute sources.",
    "Generated reports should preserve review-required flags for inferred context.",
    "Later notebooks should attach each report claim to a paper, dataset, methods page, or source URL when available.",
]

context_record = {
    **ALLEN_LAB_CONTEXT,
    "provenance_notes": PROVENANCE_NOTES,
    "review_required": True,
    "generator_track": "chatgpt",
    "notebook": "01_define_allen_lab_context.ipynb",
}

context_record


## 6. Export JSON

This creates a structured artifact for later notebooks.


In [ ]:
json_path = RESULTS_DIR / "allen_lab_context.json"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(context_record, f, indent=2, ensure_ascii=False)

print(f"Wrote {json_path}")


## 7. Export Markdown

This creates a human-readable report-context summary.


In [ ]:
def bullets(items):
    return "\n".join(f"- {item}" for item in items)

md = f"""# Allen Lab Context Profile

**Generator track:** ChatGPT  
**Notebook:** `01_define_allen_lab_context.ipynb`  
**Review required:** Yes

## Institution

{context_record['institution']}

## Likely focus areas

{bullets(context_record['likely_focus_areas'])}

## Likely equipment or platforms

{bullets(context_record['likely_equipment_or_platforms'])}

## Report priorities

{bullets(context_record['report_priorities'])}

## Provenance notes

{bullets(context_record['provenance_notes'])}

## Working interpretation

This profile defines an initial institutional context for Allen Lab report-generation experiments. It should be used as a review-aware scaffold, not as a substitute for source-specific evidence.

Later notebooks should connect this context to specific papers, datasets, methods pages, or source documents before generating public-facing lab reports.
"""

md_path = REPORTS_DIR / "allen_lab_context.md"
md_path.write_text(md, encoding="utf-8")

print(f"Wrote {md_path}")


## 8. Read exported artifacts back

This confirms that the notebook creates reusable outputs.


In [ ]:
loaded_context = json.loads(json_path.read_text(encoding="utf-8"))
print(json.dumps(loaded_context, indent=2, ensure_ascii=False))


In [ ]:
print(md_path.read_text(encoding="utf-8"))


## 9. Notebook 01 summary

Notebook 01 establishes the ChatGPT track's Allen Lab context profile.

This gives later notebooks a stable base for:

- source loading,
- paper or dataset metadata extraction,
- context-aware report section generation,
- comparison with Grok-generated outputs,
- provenance-aware labreports.app handoff demos.
